# STIC model validation

Validation of the model using field data from the ameriflux database and landsat acquisitions

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import numpy as np
import numpy.typing as npt
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sweat.common.flux import compute_et_from_le
from sweat.stic.runner import run_batch_model

### Functions

In [ ]:
# Functions


def plot_data(df: pd.DataFrame, variable: str):
    """
    Plot data
    """
    fig = px.line(
        df,
        x="date",
        y=variable,
        color="name",  # group by "name"
        markers=True,
        custom_data=["name"],
    )

    # Print value
    fig.update_traces(
        hovertemplate=(
            "<b>Site:</b> %{customdata[0]}<br>"
            "<b>Date:</b> %{x|%Y-%m-%d}<br>"
            f"<b>{variable}:</b> %{{y:.2f}}<br>"
            "<extra></extra>"
        )
    )

    # Format x-axis as YYYY-MM-DD
    fig.update_xaxes(tickformat="%Y-%m-%d")

    # Improve layout
    fig.update_layout(
        xaxis_title="Date",
        yaxis_title=variable,
        legend_title="Site",
        title={
            "text": f"Evolution of {variable} over time",
            "x": 0.5,
            "xanchor": "center",
            "font": {"size": 20},
        },
    )

    fig.show()


def plot_scatter(
    df: pd.DataFrame,
    x: str,
    y: str,
    axis_range: tuple[float, float] | None = None,
):
    """
    Interactive plot with Plotly (hover shows date)
    """

    groups = list(df.name.unique())
    groups.append("all")
    n_groups = len(groups)

    # assign one color per group (excluding "all")
    base_groups = df.name.unique()
    colors = px.colors.qualitative.Plotly

    color_map = {g: colors[i % len(colors)] for i, g in enumerate(base_groups)}

    ncols = 3
    nrows = int(np.ceil(n_groups / ncols))

    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=groups,
        horizontal_spacing=0.06,
        vertical_spacing=0.14,
    )

    for i, group in enumerate(groups):
        row = i // ncols + 1
        col = i % ncols + 1

        if group == "all":
            for g in df.name.unique():
                subset = df[df.name == g]

                fig.add_trace(
                    go.Scatter(
                        x=subset[x],
                        y=subset[y],
                        mode="markers",
                        marker={
                            "size": 6,
                            "opacity": 0.5,
                            "color": color_map[g],
                        },
                        name=g,
                        legendgroup=g,
                        showlegend=(
                            group == "all"
                        ),  # only show legend here if you want
                        customdata=subset["date"],
                        hovertemplate=(
                            "Group: " + g + "<br>"
                            f"{x}: %{{x:.3f}}<br>"
                            f"{y}: %{{y:.3f}}<br>"
                            "Date: %{customdata|%Y-%m-%d}<extra></extra>"
                        ),
                    ),
                    row=row,
                    col=col,
                )
        else:
            subset = df[df.name == group]

            xdata = subset[x]
            ydata = subset[y]
            # Scatter with hover info (including date)
            fig.add_trace(
                go.Scatter(
                    x=xdata,
                    y=ydata,
                    mode="markers",
                    marker={
                        "size": 6,
                        "opacity": 0.6,
                        "color": color_map.get(group, "gray"),
                    },
                    name=group,
                    showlegend=False,
                    customdata=subset["date"],
                    hovertemplate=(
                        f"{x}: %{{x:.3f}}<br>"
                        f"{y}: %{{y:.3f}}<br>"
                        "Date: %{customdata|%Y-%m-%d}<extra></extra>"
                    ),
                ),
                row=row,
                col=col,
            )

    fig.update_xaxes(title_text=f"{x}")
    fig.update_yaxes(title_text=f"{y}")
    if range is not None:
        fig.update_xaxes(range=axis_range)
        fig.update_yaxes(range=axis_range)
    fig.update_layout(
        height=400 * nrows,
        width=500 * ncols,
        title=f"{x} vs {y}",
    )

    fig.show()


def run_stic(df: pd.DataFrame, version: str) -> pd.DataFrame:
    """
    Compute STIC model
    """
    res = run_batch_model(data=df, threshold=0.01, nb_steps=15, version=version)
    output_df = pd.DataFrame(
        {
            "name": df["name"].values,
            "date": df["date"],
            "le": res[:, 0],
            "h": res[:, 1],
            "ef": res[:, 2],
            "g": res[:, 3],
            "rn": df["rn"].values,
            "srn": df["srn"].values,
            "ta": df["ta"].values,
            "td": df["td"].values,
            "rh": df["rh"].values,
            "ts": df["ts"].values,
            "ga": res[:, 4],
            "gs": res[:, 5],
            "t0": res[:, 6],
        }
    )
    output_df["et"] = output_df.apply(
        lambda x: compute_et_from_le(x["le"]), axis=1
    )
    output_df["rn-g"] = output_df["rn"] - output_df["g"]
    output_df["diff_le"] = output_df["le"] - df["ec_le"]
    output_df["diff_le_closed"] = output_df["le"] - df["ec_le_closed"]
    cols = list(df.columns.difference(output_df.columns))
    cols.append("date")
    cols.append("name")
    return pd.merge(left=output_df, right=df[cols], on=["date", "name"])


def compute_metrics(
    measured: npt.ArrayLike, estimated: npt.ArrayLike
) -> tuple[float, float, float, float, float]:
    """
    Compute slope, mbe, mae, rmse, r2
    """
    idx = np.isfinite(measured) & np.isfinite(estimated)
    slope, _ = np.polyfit(measured[idx], estimated[idx], 1)
    mbe = np.nanmean(estimated - measured)
    mae = mean_absolute_error(measured[idx], estimated[idx])
    rmse = np.sqrt(mean_squared_error(measured[idx], estimated[idx]))
    r2 = r2_score(measured[idx], estimated[idx])
    return (slope, mbe, mae, rmse, r2)


def generate_metrics_table(
    df: pd.DataFrame, variables: list[str] | None = None
) -> pd.DataFrame:
    """
    Generate metrics table
    """
    if variables is None:
        variables = ["le", "h", "ef", "et"]
    metrics = []
    # Compute by site
    for name, group in df.groupby("name"):
        for var in variables:
            slope, mbe, mae, rmse, r2 = compute_metrics(
                measured=group[f"ec_{var}"], estimated=group[var]
            )
            metrics.append(
                {
                    "name": name,
                    "variable": var,
                    "slope": slope,
                    "mbe": mbe,
                    "mae": mae,
                    "rmse": rmse,
                    "r2": r2,
                }
            )
    # Compute for all
    for var in variables:
        slope, mbe, mae, rmse, r2 = compute_metrics(
            measured=group[f"ec_{var}"], estimated=group[var]
        )
        metrics.append(
            {
                "name": "all",
                "variable": var,
                "slope": slope,
                "mbe": mbe,
                "mae": mae,
                "rmse": rmse,
                "r2": r2,
            }
        )
    return pd.DataFrame.from_records(metrics)


def plot_metrics(df: pd.DataFrame, variable: str = "le"):
    """
    Interactive plot with Plotly (hover shows date)
    """

    groups = list(df.name.unique())
    groups.append("all")
    n_groups = len(groups)

    # assign one color per group (excluding "all")
    base_groups = df.name.unique()
    colors = px.colors.qualitative.Plotly

    color_map = {g: colors[i % len(colors)] for i, g in enumerate(base_groups)}

    ncols = 3
    nrows = int(np.ceil(n_groups / ncols))

    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=groups,
        horizontal_spacing=0.06,
        vertical_spacing=0.14,
    )

    for i, group in enumerate(groups):
        row = i // ncols + 1
        col = i % ncols + 1

        if group == "all":
            measured = df[f"ec_{variable}"]
            estimated = df[variable]
            for g in df.name.unique():
                subset = df[df.name == g]

                fig.add_trace(
                    go.Scatter(
                        x=subset[f"ec_{variable}"],
                        y=subset[variable],
                        mode="markers",
                        marker={
                            "size": 6,
                            "opacity": 0.5,
                            "color": color_map[g],
                        },
                        name=g,
                        legendgroup=g,
                        showlegend=(
                            group == "all"
                        ),  # only show legend here if you want
                        customdata=subset["date"],
                        hovertemplate=(
                            "Group: " + g + "<br>"
                            "Measured: %{x:.3f}<br>"
                            "Estimated: %{y:.3f}<br>"
                            "Date: %{customdata|%Y-%m-%d}<extra></extra>"
                        ),
                    ),
                    row=row,
                    col=col,
                )
        else:
            subset = df[df.name == group]
            measured = subset[f"ec_{variable}"]
            estimated = subset[variable]

            # Scatter with hover info (including date)
            fig.add_trace(
                go.Scatter(
                    x=measured,
                    y=estimated,
                    mode="markers",
                    marker={
                        "size": 6,
                        "opacity": 0.6,
                        "color": color_map.get(group, "gray"),
                    },
                    name=group,
                    showlegend=False,
                    customdata=subset["date"],
                    hovertemplate=(
                        "Measured: %{x:.2f}<br>"
                        "Estimated: %{y:.2f}<br>"
                        "Date: %{customdata|%Y-%m-%d}<extra></extra>"
                    ),
                ),
                row=row,
                col=col,
            )

        # Identity line
        lims = [
            min(measured.min(), estimated.min()),
            max(measured.max(), estimated.max()),
        ]

        fig.add_trace(
            go.Scatter(
                x=lims,
                y=lims,
                mode="lines",
                line={"dash": "dash", "color": "black"},
                showlegend=False,
            ),
            row=row,
            col=col,
        )

        # Linear fit
        idx = np.isfinite(measured) & np.isfinite(estimated)
        slope, intercept = np.polyfit(measured[idx], estimated[idx], 1)
        fig.add_trace(
            go.Scatter(
                x=[0, measured.max()],
                y=[intercept, measured.max() * slope + intercept],
                mode="lines",
                line={"color": "red"},
                showlegend=False,
            ),
            row=row,
            col=col,
        )

        # Metrics
        slope_m, mbe, mae, rmse, r2 = compute_metrics(measured, estimated)

        fig.add_annotation(
            x=0.97,
            y=0.03,
            xref="x domain",
            yref="y domain",
            text=(
                f"slope = {slope_m:.2f}<br>"
                f"MBE = {mbe:.2f}<br>"
                f"MAE = {mae:.2f}<br>"
                f"RMSE = {rmse:.2f}<br>"
                f"R2 = {r2:.2f}"
            ),
            showarrow=False,
            align="right",
            row=row,
            col=col,
        )

    fig.update_xaxes(title_text=f"Measured {variable.upper()}")
    fig.update_yaxes(title_text=f"Estimated {variable.upper()}")
    fig.update_layout(
        height=400 * nrows,
        width=500 * ncols,
        title=f"Measured vs Estimated {variable.upper()}",
    )

    fig.show()

In [ ]:
if os.environ.get("SWEAT_TEST_DATA_PATH", None) is None:
    msg = "Variable SWEAT_TEST_DATA_PATH must be set"
    raise ValueError(msg)

## Input

In [ ]:
stic_data_path = os.path.join(os.environ.get("SWEAT_TEST_DATA_PATH"), "stic")
input_data_path = os.path.join(stic_data_path, "data_validation.csv")

## Read data

In [ ]:
data = pd.read_csv(input_data_path, parse_dates=["date", "date (utc)"])

In [ ]:
data

Get site names

In [ ]:
list(data["name"].unique())

### Plot data

In [ ]:
plot_scatter(data, y="nir/swir", x="ts-ta")

In [ ]:
plot_data(data, variable="ts")

In [ ]:
plot_data(data, variable="ta")

In [ ]:
plot_data(data, variable="td")

In [ ]:
plot_data(data, variable="fc")

### Compute STIC model

* Compute LE, H, EF, G, $g_{a}$, $g_{s}$,$T_{0}$ using STIC model for all input data
* Compute ET from LE

In [ ]:
results = run_stic(data, version="1.3")

In [ ]:
results

## Compute metrics

Compute the following metrics for **latent heat flux LE**, **evapotransipration ET**, **evaporative fraction EF**:

* **Slope**: corresponds to the best-fit line forced through the origin and measures how the estimated ET data scale compared to the ground-based ET data. 
* **Mean Bias Error (MBE)**: quantifies the bias in the estimated ET data relative to the ground-based ET data
* **Mean Absolute Error (MAE)**: quantifies the average absolute difference between the estimated and ground-based ET data. 
* **Root Mean Squared Error (RMSE)**: quantifies the average absolute difference between the estimated and ground-based ET data, but provides greater weight to outliers or large ‘misses’ in the estimated ET data. 
* **r-squared**: provides a measure of the proportion of the variance in the ground-based ET data that is explained or reproduced by the estimated ET data. 


In [ ]:
generate_metrics_table(results)

In [ ]:
plot_metrics(results, variable="le")

In [ ]:
plot_metrics(results, variable="h")

In [ ]:
plot_metrics(results, variable="rn-g")